# 04 — Attention Seq2Seq Training

Train and evaluate the Bahdanau attention-based encoder-decoder model.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from nmt.models.encoder import Encoder
from nmt.models.attention_seq2seq import AttentionDecoder, AttentionSeq2Seq
from nmt.training.trainer import Trainer
from nmt.training.callbacks import EarlyStopping
from nmt.training.checkpoints import CheckpointManager
from nmt.evaluation.metrics import compute_corpus_bleu
from nmt.utils.config import load_config
from nmt.utils.seed import set_seed
from nmt.utils.logging import setup_logging

setup_logging('INFO')
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Config

In [ ]:
cfg = load_config('../configs/amharic.yaml')

## 2. Build Attention Model

In [ ]:
SRC_VOCAB_SIZE = 8000
TGT_VOCAB_SIZE = 8000
PAD_IDX = 0

hidden_dim = cfg['model']['hidden_dim']
# Bidirectional encoder outputs hidden_dim * 2 features per step
encoder_dim = hidden_dim * 2 if cfg['model']['bidirectional_encoder'] else hidden_dim

encoder = Encoder(
    vocab_size=SRC_VOCAB_SIZE,
    embed_dim=cfg['model']['embed_dim'],
    hidden_dim=hidden_dim,
    num_layers=cfg['model']['num_layers'],
    dropout=cfg['model']['dropout'],
    bidirectional=cfg['model']['bidirectional_encoder'],
    padding_idx=PAD_IDX,
)
decoder = AttentionDecoder(
    vocab_size=TGT_VOCAB_SIZE,
    embed_dim=cfg['model']['embed_dim'],
    hidden_dim=hidden_dim,
    encoder_dim=encoder_dim,
    attention_dim=cfg['attention']['attention_dim'],
    num_layers=cfg['model']['num_layers'],
    dropout=cfg['model']['dropout'],
    padding_idx=PAD_IDX,
)
model = AttentionSeq2Seq(encoder, decoder, src_pad_idx=PAD_IDX, tgt_pad_idx=PAD_IDX, device=DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 3. Configure Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['training']['learning_rate'])
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    patience=cfg['training']['lr_patience'],
    factor=cfg['training']['lr_factor'],
)
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=cfg['training']['early_stopping_patience']
)
checkpoint_mgr = CheckpointManager(
    checkpoint_dir='../models/attention',
    monitor='val_loss',
    mode='min',
)
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    clip_grad_norm=cfg['training']['clip_grad_norm'],
    teacher_forcing_ratio=cfg['training']['teacher_forcing_ratio'],
    scheduler=scheduler,
    checkpoint_manager=checkpoint_mgr,
    callbacks=[early_stopping],
)

## 4. Train

> Replace `train_loader` / `val_loader` with real DataLoaders.

In [ ]:
# history = trainer.train(train_loader, val_loader, epochs=cfg['training']['epochs'])

## 5. BLEU Evaluation on Test Set

In [ ]:
# Load the best checkpoint and evaluate
# checkpoint_mgr.load_best(model)
# bleu = compute_corpus_bleu(hypotheses, references)
# print(f'Test BLEU: {bleu:.2f}')